<a href="https://colab.research.google.com/github/Tomadin/Ciencias-de-Datos/blob/main/TP5_ApacheSpark_Tomadin_Torres.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install pyspark



from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext

spark = SparkSession.builder \
        .appName("TP5_Ecommerce") \
        .getOrCreate()

print("Sesión Spark Iniciada Correctamente")


df = spark.read.csv(
    "/content/e-commerce_orders.csv",
    header=True,
    inferSchema=True,
    sep=';'
)

df.printSchema()

df.show()

df.count()



from pyspark.sql.functions import col, sum, upper, trim

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

# Elimina valores nullos
df = df.filter(df.order_approved_at.isNotNull())

# Normalizamos la columna para arreglar errores de tipeo como " PENDIENTE" o "ejecutado"
df = df.withColumn("status", upper(trim(col("status"))))

# Definimos nuestra Lista blanca
estados_validos = ["EJECUTADO", "PENDIENTE", "SUSPENDIDO"]

# Filtramos quedándonos estrictamente con los valores permitidos
df = df.filter(col("status").isin(estados_validos))


df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

df.show()


df = df.withColumn("price", df["price"].cast("float"))
df = df.withColumn("shipping_charges", df["shipping_charges"].cast("float"))


from pyspark.sql.functions import expr

df = df.withColumn("total_order_amount", expr("price + shipping_charges"))

df.show()


#  Total de ventas
df.agg({"total_order_amount": "sum"}).show()

# Promedio de ventas
df.agg({"total_order_amount": "avg"}).show()

# Cantidad de pedidos por estado
df.groupBy("status").count().show()



df.groupBy("customer_id") \
  .sum("total_order_amount") \
  .withColumnRenamed("sum(total_order_amount)", "total_spent") \
  .orderBy(col("total_spent").desc()) \
  .show(3)

Sesión Spark Iniciada Correctamente
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- shipping_charges: integer (nullable = true)
 |-- status: string (nullable = true)

+------------+------------+------------------------+-----------------+------+----------------+---------+
|    order_id| customer_id|order_purchase_timestamp|order_approved_at| price|shipping_charges|   status|
+------------+------------+------------------------+-----------------+------+----------------+---------+
|u6rPMRAYIGig|I74lXDOfoqsp|              18/11/2017|       18/11/2017|   241|             209|EJECUTADO|
|ohY8f4FEbX19|47TuLHF2s7X5|                2/6/2018|         2/6/2018|  4289|            1228|EJECUTADO|
|I28liQek73i2|dQ0dqI8Qwlj8|                8/1/2018|         9/1/2018|  5021|            6711|EJECUTADO|
|bBG1T89mlY8W|i